# 03 – Biểu diễn đặc trưng (Feature / Representation)
So sánh BoW và TF-IDF, n-gram, `min_df`, `max_df`, `max_features` – **mỗi bảng chỉ đổi một biến**, các biến khác giữ ở giá trị gốc.

In [ ]:
import sys
sys.path.insert(0, "..")  # để import được package spam_filter
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

from spam_filter.config import DataConfig
from spam_filter.data import load_dataset, split_data

cfg = DataConfig()
df = load_dataset(cfg)
X_train, X_test, y_train, y_test = split_data(df, cfg)

In [ ]:
from spam_filter.pipeline import ExperimentSpec
from spam_filter.runner import run_specs

SHOW = ["id", "cv_f1_train", "precision", "recall", "f1"]

def compare(specs):
    """Chạy các spec, trả về bảng gọn (CV trên train + test). Chọn theo cv_f1_train."""
    res, _ = run_specs(specs, X_train, y_train, X_test, y_test, cfg.cv_folds)
    return res[SHOW].round(4).set_index("id")

In [ ]:
CLEAN = {"replace_url": True, "replace_money": True, "replace_number": True, "remove_punct": True}
BASE_VEC = {"kind": "tfidf", "ngram_range": (1, 1), "min_df": 1, "max_df": 1.0, "max_features": None}

def vary(name, values, key):
    """Đổi đúng một tham số `key` của vectorizer; mọi thứ khác = BASE_VEC."""
    specs = [ExperimentSpec(id=f"{name}={v}", description=f"{name}={v}", model="nb",
                            model_params={"alpha": 0.1}, cleaner=CLEAN,
                            vectorizer={**BASE_VEC, key: v}) for v in values]
    return compare(specs)

print("Cố định: TextCleaner(URL/tiền/số/dấu câu), MultinomialNB alpha=0.1")
print("Gốc    :", BASE_VEC)

## 1. Phương pháp: BoW vs TF-IDF

In [ ]:
vary("kind", ["bow", "tfidf"], "kind")

## 2. n-gram

In [ ]:
vary("ngram", [(1, 1), (1, 2), (1, 3)], "ngram_range")

## 3. `min_df`

In [ ]:
vary("min_df", [1, 2, 3, 5], "min_df")

## 4. `max_df`

In [ ]:
vary("max_df", [0.5, 0.8, 0.9, 1.0], "max_df")

## 5. Số đặc trưng tối đa `max_features`

In [ ]:
vary("max_features", [500, 1000, 3000, None], "max_features")

## 6. Kích thước từ vựng tương ứng

In [ ]:
from spam_filter.features import build_vectorizer
from spam_filter.preprocessing import TextCleaner
cleaned = TextCleaner(**CLEAN).transform(X_train)
pd.DataFrame([{"ngram": ng, "min_df": m, "n_features": len(build_vectorizer("tfidf", ng, min_df=m).fit(cleaned).vocabulary_)}
              for ng in [(1, 1), (1, 2), (1, 3)] for m in (1, 2, 5)])

## Nhận xét (nhóm điền)
Biến nào ảnh hưởng nhiều nhất? n-gram có đáng với chi phí tăng số đặc trưng không? Chọn cấu hình nào cho bước Model và vì sao (căn cứ `cv_f1_train`)?